# Notebook 15: Final prediction audit and paper outputs

Run the **one executable cell on CPU** after Notebook 14 completes. It uses your saved predictions and candidate caches. It needs **no GPU, model fitting, policy inference, new action selection or threshold changes**. The notebook does not improve or replace the frozen method. It checks and explains the existing results, including stronger baselines and failures.

Keep the existing folders in `MyDrive/SensorRepair_ICDSIS`: prepared `processed/v1`, candidate caches in Notebook 02/04/05, and Notebook 09/12/13/14 outputs. No dataset downloads or manual uploads are needed. Select **Runtime → Change runtime type → CPU**, then run the cell. If a 60-minute budget pause occurs, rerun the same cell. Completed per-configuration audits are reused.

The audit covers **DSADS four folds × three seeds and HAR's official test split × three seeds**, with **28 methods**. The primary method remains `source_mean_f1`. It checks hashes, label/window/participant alignment, every saved prediction metric, the mapping between saved sensor actions and cached candidate predictions, all Notebook 14 aggregate/seed/comparison/harm tables, and reported coverage. It reads only the prepared protocol and split indices, rather than the raw signal arrays or model checkpoints. Imported seed-2026 predictions are located automatically in their original Notebook 12/13 folders; the ten newer configurations come from Notebook 14.

## Primary and secondary results

- **Primary:** condition-level Macro-F1 averaged within each scope, then equally across required folds within seed, then equally across three seeds. The seed standard deviation is descriptive. All 28 methods and all original comparisons are retained.
- **Secondary:** for each participant, average that participant's present-class Macro-F1 over the scope's conditions and three seeds, then weight participants equally. Bootstrap **whole participants**, paired across methods, 10,000 times with a fixed seed. All repeated windows, fault views and seeds stay inside the participant. DSADS has **eight**, HAR **nine**, unique participants. These percentile intervals are conditional on the saved fits and the fixed evaluation design. They do not cover training randomness, dependence from shared training data, selection after test inspection or the primary score. They are not significance tests or population-risk guarantees. No multiplicity-adjusted inferential claim is made.

## Diagnostics and manuscript files

The notebook saves fault-family/severity curves, participant differences, correct-to-incorrect harm, all harmful cases of the primary method, and keep/mask/reconstruct action diagnostics. Action rates and prediction-change rates are distinct. Action-wise helpful/harmful counts describe accuracy transitions; Macro-F1 is not decomposed additively by action.

It creates:

- Main, ablation and secondary-interval tables in CSV and LaTeX. The stronger HAR fusion/stacking baselines remain visible. Full tables include every method.
- Three vector PDF figures with 300-dpi PNG copies: fault robustness, F1–harm tradeoff, and participant differences versus the matched window ensemble. Participant error bars show the range across three seeds, not confidence intervals.
- Participant, fault, action, harm and case-level diagnostics, plus a methods/limitations note and a reproducible analysis plan.
- A small ZIP paper bundle with tables, figures and notes. Detailed prediction/case files remain outside this bundle.

The same test outcomes have already been inspected. The entire study remains exploratory. Stuck faults were excluded from fitting but their earlier outcomes were inspected. Harm guards were empirical validation criteria; test harm may exceed them. The source jackknife excludes controller outcome rows, rather than refitting the entire signal pipeline separately inside each jackknife block. No claim of universal superiority, certified robustness, novelty or acceptance is made.

Outputs: `SensorRepair_ICDSIS/outputs/notebook15_v1`.

After **NOTEBOOK 15 COMPLETE**, share the final output and:

- `notebook15_comparison_summary.csv`
- `notebook15_harm_audit.csv`
- `notebook15_action_family_summary.csv`
- `notebook15_run_report.json`
- `notebook15_paper_bundle.zip`


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import hashlib
import json
import os
import shutil
import time
import zipfile
import zlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
OUTPUT=PROJECT/'outputs'/'notebook15_v1'
LOCAL=Path('/content/sensorrepair_notebook15_v1')
CODE_VERSION='sensorrepair_nb15_v1'
CELL_SIGNATURE='06de0b16933b884ad778f9d5c467fd7de3f9d3d9905e50d82e9fff940f257ce1'
BOOTSTRAP_RESAMPLES=10000
BOOTSTRAP_SEED=20261022
CONFIDENCE_LEVEL=.95
SESSION_MINUTES=60
SEEDS=[2026,2027,2028]
JOBS=[]
for seed in SEEDS:
    JOBS += [{'dataset':'dsads','fold':f'fold_{f}','seed':seed} for f in range(4)]
    JOBS += [{'dataset':'har','fold':'fold_0','seed':seed}]
DEV_CONDITIONS=['clean']+[f'{f}_{i}' for f in ['dropout','bias','gain'] for i in [1,2,3]]
CONDITIONS=DEV_CONDITIONS+[f'stuck_{i}' for i in [1,2,3]]
SCOPES={'clean':['clean'],'known_faults':DEV_CONDITIONS[1:],'stuck_excluded_from_fitting':CONDITIONS[10:]}
NEW_METHODS=['oof_accuracy_utility','oof_f1_utility','source_mean_f1','source_min_f1','window_mean_f1','window_min_f1']
OLD_METHODS=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct',
 'confidence_select','oracle_diagnostic','harm_aware_policy','benefit_only_policy','confidence_margin_tuned',
 'residual_mask_tuned','residual_reconstruct_tuned','ternary_outcome_selector','utility_regression_selector',
 'uniform_fusion_safe','uniform_fusion_accuracy','global_fusion_safe','global_fusion_accuracy',
 'gated_fusion_safe','gated_fusion_accuracy','residual_stacking_safe','residual_stacking_accuracy']
METHODS=OLD_METHODS+NEW_METHODS
PRIMARY='source_mean_f1'
METRICS=['macro_f1','accuracy','decision_change_rate','helpful_rate','harmful_rate']
COMPARATORS=[m for m in METHODS if m not in [PRIMARY,'oracle_diagnostic']]
DISPLAY_METHODS=['keep','residual_gate_mask','confidence_select','harm_aware_policy',
 'oof_accuracy_utility','oof_f1_utility','window_mean_f1','source_min_f1',
 'gated_fusion_safe','residual_stacking_safe','residual_stacking_accuracy',PRIMARY]
DISPLAY_COMPARATORS=['keep','harm_aware_policy','oof_accuracy_utility','oof_f1_utility',
 'window_mean_f1','gated_fusion_safe','residual_stacking_safe']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
SESSION_DEADLINE=time.monotonic()+SESSION_MINUTES*60
print('NOTEBOOK 15 | final saved-prediction audit and paper outputs | CPU | one executable cell',flush=True)
print('Both datasets, all 15 configurations, all 28 saved methods. No training, model inference or threshold changes.',flush=True)
print('Primary aggregation unchanged. Participant-bootstrap intervals are secondary and conditional on saved fits.',flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()



def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()



def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)



def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)



def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)



def identity(job):
    return f'{job["dataset"]}_{job["fold"]}_seed_{job["seed"]}'



def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'



def verified_copy(source,relative,expected):
    assert not Path(relative).is_absolute() and '..' not in Path(relative).parts
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(source,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {source}'
    os.replace(partial,target)
    return target



def stage_json(number,name):
    root=PROJECT/'outputs'/f'notebook{number:02d}_v1'
    inv={r['filename']:r for r in json.loads((root/f'notebook{number:02d}_artifact_inventory.json').read_text())}
    path=verified_copy(root/name,f'nb{number:02d}/{name}',inv[name]['sha256'])
    return json.loads(path.read_text()),inv[name]['sha256']



def condition_rows(job,y,predictions,conditions,classes,actions=None):
    base=predictions[0];rows=[]
    assert predictions.shape==(len(METHODS),len(conditions),len(y))
    for m,method in enumerate(METHODS):
        for d,condition in enumerate(conditions):
            row={**job,'method':method,'condition':condition,'windows':len(y),**metrics(y,predictions[m,d],base[d],classes)}
            row['sensor_action_intervention_rate']=(float((actions[NEW_METHODS.index(method),d]!=0).mean())
                 if actions is not None and method in NEW_METHODS else (0. if method=='keep' else np.nan))
            rows.append(row)
    return pd.DataFrame(rows)



def participant_rows(job,y,subjects,predictions,classes):
    rows=[]
    for m,method in enumerate(METHODS):
        if method=='oracle_diagnostic':continue
        for person in np.unique(subjects):
            use=subjects==person;present=np.unique(y[use])
            for scope,conditions in SCOPES.items():
                values=[metrics(y[use],predictions[m,CONDITIONS.index(c),use],predictions[0,CONDITIONS.index(c),use],classes,present) for c in conditions]
                average={k:float(np.mean([v[k] for v in values])) for k in METRICS}
                average['macro_f1_present_classes']=average.pop('macro_f1')
                rows.append({**job,'method':method,'subject':int(person),'scope':scope,'windows':int(use.sum()),
                    'classes_present':len(present),**average})
    return pd.DataFrame(rows)



def build_audits(summary,conditions,participants):
    comparisons=[];harm=[]
    for (dataset,fold,seed,scope),group in summary.groupby(['dataset','fold','seed','scope'],sort=False):
        lookup={r['method']:r for r in group.to_dict('records')}
        for focal in [PRIMARY,'source_min_f1','oof_f1_utility']:
            for comparator,row in lookup.items():
                if comparator in [focal,'oracle_diagnostic']:continue
                comparisons.append({'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':focal,
                  'comparator':comparator,'primary_method':focal==PRIMARY,
                  'policy_macro_f1_pct':lookup[focal]['macro_f1']*100,'comparator_macro_f1_pct':row['macro_f1']*100,
                  'delta_macro_f1_pp':(lookup[focal]['macro_f1']-row['macro_f1'])*100,
                  'policy_harmful_rate_pct':lookup[focal]['harmful_rate']*100,
                  'comparator_harmful_rate_pct':row['harmful_rate']*100})
        for method,row in lookup.items():
            part=conditions[(conditions.dataset==dataset)&(conditions.fold==fold)&(conditions.seed==seed)&(conditions.method==method)&conditions.condition.isin(SCOPES[scope])]
            people=participants[(participants.dataset==dataset)&(participants.fold==fold)&(participants.seed==seed)&(participants.method==method)&(participants.scope==scope)]
            record={'dataset':dataset,'fold':fold,'seed':seed,'scope':scope,'method':method,
              'macro_f1_pct':row['macro_f1']*100,'mean_harm_pct':row['harmful_rate']*100,
              'mean_harm_le_1pct':bool(row['harmful_rate']<=.01+1e-12),
              'maximum_condition_harm_pct':float(part.harmful_rate.max()*100),
              'conditions_exceeding_1pct_harm':int((part.harmful_rate>.01+1e-12).sum()),
              'worst_condition_macro_f1_pct':float(part.macro_f1.min()*100),
              'maximum_participant_mean_harm_pct':float(people.harmful_rate.max()*100) if len(people) else np.nan,
              'participants_exceeding_1pct_mean_harm':int((people.harmful_rate>.01+1e-12).sum()) if len(people) else np.nan,
              'test_participants':len(people),'descriptive_only':True}
            if scope=='clean':
                loss=(lookup['keep']['macro_f1']-row['macro_f1'])*100
                record.update(clean_f1_loss_vs_keep_pp=loss,clean_f1_loss_le_0_5pp=bool(loss<=.5+1e-12))
            harm.append(record)
    return pd.DataFrame(comparisons),pd.DataFrame(harm)



def metrics(y,pred,base,K,present=None):
    cm=np.bincount(y*K+pred,minlength=K*K).reshape(K,K)
    den=cm.sum(0)+cm.sum(1)
    f1=np.divide(2.*cm.diagonal(),den,out=np.zeros(K),where=den>0)
    correct=pred==y;was_correct=base==y
    helpful=float((~was_correct & correct).mean());harmful=float((was_correct & ~correct).mean())
    assert abs(correct.mean()-was_correct.mean()-helpful+harmful)<1e-12
    return {'macro_f1':float(f1.mean() if present is None else f1[present].mean()),'accuracy':float(correct.mean()),
            'decision_change_rate':float((pred!=base).mean()),'helpful_rate':helpful,'harmful_rate':harmful}



class PauseAudit(Exception):
    pass


def check_budget():
    if time.monotonic()>=SESSION_DEADLINE:
        raise PauseAudit('Session budget reached. Rerun the same CPU cell; completed configuration audits are saved.')


def table14(name):
    filename='notebook14_'+name+'.csv'
    return pd.read_csv(verified_copy(SOURCE14/filename,'nb14/'+filename,REFERENCE_HASHES[name]))


def run_file(job,name):
    item=INPUT_RUNS[identity(job)]
    record=next(a for a in item['record']['artifacts'] if a['filename']==name)
    rel='runs/'+identity(job)+'/'+name
    return verified_copy(ROOTS[item['stage']]/rel,f'nb{item["stage"]:02d}/'+rel,record['sha256'])


def assert_frame(computed,expected,keys,numeric,flags=()):
    assert_table_parity(computed,expected,keys,numeric)
    a=computed.set_index(keys).sort_index();b=expected.set_index(keys).sort_index()
    for name in flags:
        left=a[name].fillna('missing').astype(str).to_numpy()
        right=b[name].fillna('missing').astype(str).to_numpy()
        assert np.array_equal(left,right),f'Flag/metadata mismatch: {name}'


def check_metadata(job,arrays):
    y=arrays['y'];subjects=arrays['subjects'];indices=arrays['source_indices'];ids=arrays['window_ids']
    assert y.ndim==subjects.ndim==indices.ndim==ids.ndim==1
    assert len(y)==len(subjects)==len(indices)==len(ids)>0
    assert np.issubdtype(y.dtype,np.integer) and y.min()>=0
    assert y.max()<len(protocol['datasets'][job['dataset']]['class_names'])
    assert len(np.unique(ids))==len(y)==len(np.unique(indices))
    expected=INPUT_RUNS[identity(job)]['record']
    assert len(y)==expected['test_windows']
    assert np.unique(subjects).tolist()==expected['test_participants']
    name=job['dataset']+'_splits.npz'
    path=verified_copy(PREPARED/name,'prepared/'+name,prepared_inventory[name]['sha256'])
    with np.load(path,allow_pickle=False) as z:
        assert np.array_equal(indices,z[job['fold']+'_test'])
        for split in ['train','policy','validation']:
            assert not np.intersect1d(indices,z[job['fold']+'_'+split]).size
    key=(job['dataset'],job['fold'])
    meta={k:np.array(arrays[k],copy=True) for k in ['y','subjects','source_indices','window_ids']}
    if key in ANCHORS:
        for k in meta:assert np.array_equal(meta[k],ANCHORS[key][k]),f'Cross-seed window alignment changed: {key}, {k}'
    else:
        assert job['seed']==SEEDS[0]
        ANCHORS[key]=meta


def verify_candidate_mapping(job,arrays):
    ds=job['dataset'];groups=protocol['datasets'][ds]['sensor_groups'];G=len(groups)
    expected_names=['keep']+[f'mask:{g["name"]}' for g in groups]+[f'reconstruct:{g["name"]}' for g in groups]
    actions=arrays['new_actions'];pred=arrays['predictions'];N=len(arrays['y']);K=len(protocol['datasets'][ds]['class_names'])
    assert actions.shape==(len(NEW_METHODS),len(CONDITIONS),N)
    assert np.issubdtype(actions.dtype,np.integer) and actions.min()>=0 and actions.max()<=2*G
    assert arrays['new_action_methods'].tolist()==NEW_METHODS
    for d,condition in enumerate(CONDITIONS):
        item=CACHE_INPUTS[identity(job)]['test|'+condition]
        path=verified_copy(STAGES[item['stage']]/item['filename'],item['stage']+'/'+item['filename'],item['sha256'])
        with np.load(path,allow_pickle=False) as z:
            assert z['signature'].item()==item['signature']
            assert z['condition'].item()==condition and z['split'].item()=='test'
            assert z['action_names'].tolist()==expected_names
            for name in ['y','subjects','window_ids','source_indices']:assert np.array_equal(z[name],arrays[name])
            p=z['probabilities']
            assert p.shape==(N,1+2*G,K) and np.isfinite(p).all() and (p>=0).all()
            assert np.allclose(p.sum(-1),1.,atol=1e-5)
            candidates=p.argmax(-1)
        assert np.array_equal(candidates[:,0],pred[METHODS.index('keep'),d])
        for m,method in enumerate(NEW_METHODS):
            selected=candidates[np.arange(N),actions[m,d]]
            assert np.array_equal(selected,pred[METHODS.index(method),d]),f'Saved action/prediction mismatch: {identity(job)}, {condition}, {method}'


def action_diagnostics(job,arrays):
    rows=[];cases=[];y=arrays['y'];N=len(y)
    groups=protocol['datasets'][job['dataset']]['sensor_groups'];G=len(groups)
    labels=protocol['datasets'][job['dataset']]['class_names']
    for m,method in enumerate(NEW_METHODS):
        predicted=arrays['predictions'][METHODS.index(method)]
        keep=arrays['predictions'][METHODS.index('keep')]
        choices=arrays['new_actions'][m]
        for d,condition in enumerate(CONDITIONS):
            was_correct=keep[d]==y;now_correct=predicted[d]==y
            helpful=~was_correct&now_correct;harmful=was_correct&~now_correct
            changed=predicted[d]!=keep[d]
            total_help=total_harm=0
            for action in range(1+2*G):
                use=choices[d]==action;n=int(use.sum())
                family='keep' if action==0 else ('mask' if action<=G else 'reconstruct')
                group='none' if action==0 else groups[(action-1)%G]['name']
                fix=int((use&helpful).sum());harm=int((use&harmful).sum());change=int((use&changed).sum())
                total_help+=fix;total_harm+=harm
                rows.append({**job,'method':method,'condition':condition,'action_id':action,'action_family':family,
                  'sensor_group':group,'windows':N,'action_count':n,'helpful_count':fix,'harmful_count':harm,
                  'class_change_count':change,'action_rate':n/N,'helpful_rate_all_windows':fix/N,
                  'harmful_rate_all_windows':harm/N,'class_change_rate_all_windows':change/N,
                  'helpful_given_action':fix/n if n else np.nan,'harmful_given_action':harm/n if n else np.nan,
                  'unchanged_class_given_action':1.-change/n if n else np.nan})
            assert total_help==int(helpful.sum()) and total_harm==int(harmful.sum())
            if method==PRIMARY:
                for i in np.flatnonzero(harmful):
                    action=int(choices[d,i]);assert action!=0
                    cases.append({**job,'condition':condition,'subject':int(arrays['subjects'][i]),
                      'window_id':str(arrays['window_ids'][i]),'source_index':int(arrays['source_indices'][i]),
                      'true_class':int(y[i]),'true_class_name':str(labels[int(y[i])]),
                      'keep_prediction':int(keep[d,i]),'policy_prediction':int(predicted[d,i]),
                      'policy_class_name':str(labels[int(predicted[d,i])]),'action_id':action,
                      'action_family':'mask' if action<=G else 'reconstruct','sensor_group':groups[(action-1)%G]['name']})
    columns=['dataset','fold','seed','condition','subject','window_id','source_index','true_class','true_class_name',
      'keep_prediction','policy_prediction','policy_class_name','action_id','action_family','sensor_group']
    return pd.DataFrame(rows),pd.DataFrame(cases,columns=columns)


def audit_one(job):
    run=OUTPUT/'runs'/identity(job);run.mkdir(parents=True,exist_ok=True)
    done=run/'audit_complete.json'
    if done.exists():
        result=json.loads(done.read_text());assert result['analysis_signature']==ANALYSIS_SIGNATURE
        for item in result['artifacts']:assert file_sha256(run/item['filename'])==item['sha256']
        with np.load(run/'window_metadata.npz',allow_pickle=False) as z:check_metadata(job,z)
        print('  Reused audit:',identity(job),flush=True)
        return result
    check_budget()
    path=run_file(job,'test_predictions.npz')
    with np.load(path,allow_pickle=False) as z:
        needed=['methods','conditions','predictions','new_action_methods','new_actions','y','subjects','window_ids','source_indices','study_signature']
        arrays={name:z[name] for name in needed}
    assert arrays['study_signature'].item()==INPUT_RUNS[identity(job)]['record']['study_signature']
    assert arrays['methods'].tolist()==METHODS and arrays['conditions'].tolist()==CONDITIONS
    check_metadata(job,arrays)
    N=len(arrays['y']);K=len(protocol['datasets'][job['dataset']]['class_names'])
    pred=arrays['predictions']
    assert pred.shape==(len(METHODS),len(CONDITIONS),N) and np.issubdtype(pred.dtype,np.integer)
    assert pred.min()>=0 and pred.max()<K
    verify_candidate_mapping(job,arrays)
    rows=condition_rows(job,arrays['y'],pred,CONDITIONS,K,arrays['new_actions'])
    participants=participant_rows(job,arrays['y'],arrays['subjects'],pred,K)
    summaries=[]
    for scope,cs in SCOPES.items():
        frame=rows[rows.condition.isin(cs)].groupby(['dataset','fold','seed','method'],sort=False)[METRICS+['sensor_action_intervention_rate']].mean().reset_index()
        frame['scope']=scope;summaries.append(frame)
    summary=pd.concat(summaries,ignore_index=True)
    filt=lambda x:x[(x.dataset==job['dataset'])&(x.fold==job['fold'])&(x.seed==job['seed'])]
    assert_frame(rows,filt(REFERENCE['per_condition']),['dataset','fold','seed','method','condition'],METRICS+['sensor_action_intervention_rate'])
    assert_frame(summary,filt(REFERENCE['test_summary']),['dataset','fold','seed','method','scope'],METRICS+['sensor_action_intervention_rate'])
    assert_frame(participants,filt(REFERENCE['per_participant']),['dataset','fold','seed','method','scope','subject'],
        ['macro_f1_present_classes','accuracy','decision_change_rate','helpful_rate','harmful_rate','windows','classes_present'])
    actions,cases=action_diagnostics(job,arrays)
    for name,value in [('per_condition',rows),('per_participant',participants),('test_summary',summary),
                       ('action_audit',actions),('harmful_cases',cases)]:save_csv(run/(name+'.csv'),value)
    save_npz(run/'window_metadata.npz',**{name:arrays[name] for name in ['y','subjects','window_ids','source_indices']})
    names=['per_condition.csv','per_participant.csv','test_summary.csv','action_audit.csv','harmful_cases.csv','window_metadata.npz']
    result={**job,'analysis_signature':ANALYSIS_SIGNATURE,'prediction_source_stage':INPUT_RUNS[identity(job)]['stage'],
      'test_windows':N,'test_participants':np.unique(arrays['subjects']).tolist(),'methods_reproduced':len(METHODS),
      'candidate_action_mapping_verified':True,'original_metrics_reproduced':True,
      'artifacts':[{'filename':name,'sha256':file_sha256(run/name)} for name in names]}
    save_json(done,result)
    print(f'  Reproduced {identity(job)}: {N} windows; {len(result["test_participants"])} participants; saved actions verified.',flush=True)
    del arrays,pred,rows,summary,participants,actions,cases;gc.collect()
    return result


def reproduce_aggregates(results):
    frames={name:pd.concat([pd.read_csv(OUTPUT/'runs'/identity(r)/(name+'.csv')) for r in results],ignore_index=True)
            for name in ['test_summary','per_condition','per_participant','action_audit','harmful_cases']}
    summary=frames['test_summary'];conditions=frames['per_condition'];participants=frames['per_participant']
    coverage=[]
    for ds in ['dsads','har']:
        required=4 if ds=='dsads' else 1
        for seed in SEEDS:
            current=[r for r in results if r['dataset']==ds and r['seed']==seed]
            assert len(current)==required
            seen=set()
            for r in current:
                p=set(r['test_participants']);assert not seen&p;seen|=p
            assert len(seen)==(8 if ds=='dsads' else 9)
            coverage.append({'dataset':ds,'seed':seed,'completed_folds':len(current),'required_folds':required,
              'full_fold_coverage':True,'unique_test_participants':len(seen),'unique_test_windows':sum(r['test_windows'] for r in current)})
    coverage=pd.DataFrame(coverage)
    columns=METRICS+['sensor_action_intervention_rate']
    per_seed=summary.groupby(['dataset','seed','method','scope'],sort=False)[columns].mean().reset_index().merge(
        coverage,on=['dataset','seed'],validate='many_to_one')
    aggregate=per_seed.groupby(['dataset','method','scope'],sort=False)[columns].mean().reset_index()
    spread=per_seed.groupby(['dataset','method','scope'],sort=False)[METRICS].std(ddof=1).reset_index().rename(
        columns={m:m+'_seed_std' for m in METRICS})
    aggregate=aggregate.merge(spread,on=['dataset','method','scope'],validate='one_to_one')
    cov=[]
    for ds,group in coverage.groupby('dataset',sort=False):
        assert group.unique_test_participants.nunique()==group.unique_test_windows.nunique()==1
        cov.append({'dataset':ds,'complete_seeds':len(group),'required_seeds':len(SEEDS),'full_seed_coverage':True,
          'complete_configurations':int(group.completed_folds.sum()),'required_configurations':int(group.required_folds.sum()),
          'unique_test_participants':int(group.unique_test_participants.iloc[0]),'unique_test_windows':int(group.unique_test_windows.iloc[0]),
          'included_seeds':json.dumps(group.seed.astype(int).tolist())})
    cov=pd.DataFrame(cov);aggregate=aggregate.merge(cov,on='dataset',validate='many_to_one')
    comparison,harm=build_audits(summary,conditions,participants)
    seed_rows=[]
    for (ds,seed,method,comparator,scope),group in comparison.groupby(['dataset','seed','method','comparator','scope'],sort=False):
        c=coverage[(coverage.dataset==ds)&(coverage.seed==seed)].iloc[0]
        seed_rows.append({'dataset':ds,'seed':int(seed),'method':method,'comparator':comparator,'scope':scope,
          'primary_method':method==PRIMARY,'completed_folds':int(c.completed_folds),'required_folds':int(c.required_folds),'full_fold_coverage':True,
          **{k:float(group[k].mean()) for k in ['policy_macro_f1_pct','comparator_macro_f1_pct','delta_macro_f1_pp','policy_harmful_rate_pct','comparator_harmful_rate_pct']},
          'positive_folds':int((group.delta_macro_f1_pp>1e-12).sum()),'negative_folds':int((group.delta_macro_f1_pp< -1e-12).sum()),
          'tied_folds':int((group.delta_macro_f1_pp.abs()<=1e-12).sum())})
    per_seed_comparison=pd.DataFrame(seed_rows);overall=[]
    for (ds,method,comparator,scope),group in per_seed_comparison.groupby(['dataset','method','comparator','scope'],sort=False):
        runs=comparison[(comparison.dataset==ds)&(comparison.method==method)&(comparison.comparator==comparator)&(comparison.scope==scope)]
        c=cov[cov.dataset==ds].iloc[0]
        overall.append({'dataset':ds,'method':method,'comparator':comparator,'scope':scope,'primary_method':method==PRIMARY,
          'complete_seeds':len(group),'required_seeds':len(SEEDS),'full_seed_coverage':True,'configurations':len(runs),
          'unique_test_participants':int(c.unique_test_participants),
          **{k:float(group[k].mean()) for k in ['policy_macro_f1_pct','comparator_macro_f1_pct','delta_macro_f1_pp','policy_harmful_rate_pct','comparator_harmful_rate_pct']},
          'delta_seed_std_pp':float(group.delta_macro_f1_pp.std(ddof=1)),
          'positive_seeds':int((group.delta_macro_f1_pp>1e-12).sum()),'negative_seeds':int((group.delta_macro_f1_pp< -1e-12).sum()),
          'tied_seeds':int((group.delta_macro_f1_pp.abs()<=1e-12).sum()),
          'positive_runs':int((runs.delta_macro_f1_pp>1e-12).sum()),'negative_runs':int((runs.delta_macro_f1_pp< -1e-12).sum()),
          'tied_runs':int((runs.delta_macro_f1_pp.abs()<=1e-12).sum())})
    overall=pd.DataFrame(overall)
    checks=[('coverage',coverage,['dataset','seed']),('dataset_coverage',cov,['dataset']),
      ('per_seed_summary',per_seed,['dataset','seed','method','scope']),('aggregate_summary',aggregate,['dataset','method','scope']),
      ('per_fold_comparisons',comparison,['dataset','fold','seed','method','comparator','scope']),
      ('per_seed_comparisons',per_seed_comparison,['dataset','seed','method','comparator','scope']),
      ('comparisons',overall,['dataset','method','comparator','scope']),('harm_audit',harm,['dataset','fold','seed','method','scope'])]
    for name,computed,keys in checks:
        ref=REFERENCE[name]
        flags=[c for c in ref if c not in keys and (ref[c].dtype=='object' or ref[c].dtype=='bool')]
        numeric=[c for c in ref if c not in keys+flags]
        assert_frame(computed,ref,keys,numeric,flags)
        save_csv(OUTPUT/f'notebook15_{name}.csv',computed)
    diagnostics=[]
    for row in aggregate[aggregate.method==PRIMARY].to_dict('records'):
        x=harm[(harm.dataset==row['dataset'])&(harm.method==PRIMARY)&(harm.scope==row['scope'])]
        diagnostics.append({'dataset':row['dataset'],'scope':row['scope'],'complete_seeds':len(SEEDS),
          'macro_f1_pct':row['macro_f1']*100,'mean_harm_pct':row['harmful_rate']*100,'runs':len(x),
          'runs_exceeding_1pct_mean_harm':int((~x.mean_harm_le_1pct).sum()),
          'maximum_condition_harm_pct':float(x.maximum_condition_harm_pct.max()),
          'maximum_participant_scope_harm_pct':float(x.maximum_participant_mean_harm_pct.max()),'descriptive_only':True})
    diagnostics=pd.DataFrame(diagnostics)
    assert_frame(diagnostics,REFERENCE['primary_diagnostics'],['dataset','scope'],
        [c for c in diagnostics if c not in ['dataset','scope','descriptive_only']],['descriptive_only'])
    save_csv(OUTPUT/'notebook15_primary_diagnostics.csv',diagnostics)
    for name,frame in frames.items():save_csv(OUTPUT/f'notebook15_{name}.csv',frame)
    return frames,aggregate,per_seed,overall,harm,cov,diagnostics


def participant_bootstrap(participants,primary_comparisons):
    keys=['dataset','method','scope','subject']
    counts=participants.groupby(keys,sort=False).size()
    assert (counts==len(SEEDS)).all()
    balanced=participants.groupby(keys,sort=False)['macro_f1_present_classes'].mean().reset_index()
    rows=[];saved_draws={}
    q=(1.-CONFIDENCE_LEVEL)/2.
    for ds in ['dsads','har']:
        persons=np.sort(balanced[balanced.dataset==ds].subject.unique())
        assert len(persons)==(8 if ds=='dsads' else 9)
        rng=np.random.default_rng(BOOTSTRAP_SEED+zlib.crc32(ds.encode()))
        draws=rng.integers(0,len(persons),size=(BOOTSTRAP_RESAMPLES,len(persons)))
        weights=np.eye(len(persons))[draws].mean(axis=1)
        saved_draws[ds+'_subjects']=persons;saved_draws[ds+'_draws']=draws.astype(np.int16)
        for scope in SCOPES:
            pivot=balanced[(balanced.dataset==ds)&(balanced.scope==scope)].pivot(index='subject',columns='method',values='macro_f1_present_classes').loc[persons]
            assert set(pivot.columns)==set(METHODS)-{'oracle_diagnostic'} and np.isfinite(pivot.to_numpy()).all()
            for comparator in COMPARATORS:
                difference=pivot[PRIMARY].to_numpy()-pivot[comparator].to_numpy()
                distribution=weights@difference
                lo,hi=np.quantile(distribution,[q,1.-q])
                row=primary_comparisons[(primary_comparisons.dataset==ds)&(primary_comparisons.scope==scope)&(primary_comparisons.comparator==comparator)].iloc[0].to_dict()
                row.update(secondary_participants=len(persons),secondary_policy_macro_f1_pct=float(pivot[PRIMARY].mean()*100),
                  secondary_comparator_macro_f1_pct=float(pivot[comparator].mean()*100),
                  secondary_participant_balanced_delta_pp=float(difference.mean()*100),
                  secondary_ci_low_pp=float(lo*100),secondary_ci_high_pp=float(hi*100),
                  secondary_positive_participants=int((difference>1e-12).sum()),
                  secondary_negative_participants=int((difference< -1e-12).sum()),
                  secondary_tied_participants=int((np.abs(difference)<=1e-12).sum()),
                  bootstrap_resamples=BOOTSTRAP_RESAMPLES,conditional_on_saved_fits=True,
                  secondary_interval_applies_to_primary=False,significance_claim=False)
                rows.append(row)
    save_npz(OUTPUT/'notebook15_participant_bootstrap_draws.npz',analysis_signature=np.asarray(ANALYSIS_SIGNATURE),**saved_draws)
    result=pd.DataFrame(rows)
    save_csv(OUTPUT/'notebook15_comparison_summary.csv',result)
    save_csv(OUTPUT/'notebook15_participant_balanced_scores.csv',balanced)
    return result,balanced


def diagnostic_summaries(frames,harm):
    rows=frames['per_condition'].copy()
    rows['fault_family']=rows.condition.map(lambda c:'clean' if c=='clean' else c.rsplit('_',1)[0])
    rows['severity_level']=rows.condition.map(lambda c:0 if c=='clean' else int(c.rsplit('_',1)[1]))
    cols=METRICS+['sensor_action_intervention_rate']
    by_seed=rows.groupby(['dataset','seed','method','fault_family','severity_level'],sort=False)[cols].mean().reset_index()
    by_severity=by_seed.groupby(['dataset','method','fault_family','severity_level'],sort=False)[cols].mean().reset_index()
    by_family=by_severity.groupby(['dataset','method','fault_family'],sort=False)[cols].mean().reset_index()
    actions=frames['action_audit']
    family=actions.groupby(['dataset','fold','seed','method','condition','action_family'],sort=False)[
        ['action_count','helpful_count','harmful_count','class_change_count']].sum().reset_index()
    denominator=actions.groupby(['dataset','fold','seed','method','condition'],sort=False).windows.first().reset_index()
    family=family.merge(denominator,on=['dataset','fold','seed','method','condition'],validate='many_to_one')
    for target,count in [('action_rate','action_count'),('helpful_rate_all_windows','helpful_count'),
                         ('harmful_rate_all_windows','harmful_count'),('class_change_rate_all_windows','class_change_count')]:
        family[target]=family[count]/family.windows
    family_rows=[]
    rates=['action_rate','helpful_rate_all_windows','harmful_rate_all_windows','class_change_rate_all_windows']
    for scope,cs in SCOPES.items():
        run=family[family.condition.isin(cs)].groupby(['dataset','fold','seed','method','action_family'],sort=False)[rates].mean().reset_index()
        seed=run.groupby(['dataset','seed','method','action_family'],sort=False)[rates].mean().reset_index()
        average=seed.groupby(['dataset','method','action_family'],sort=False)[rates].mean().reset_index();average['scope']=scope
        family_rows.append(average)
    family_summary=pd.concat(family_rows,ignore_index=True)
    assert np.allclose(family_summary.groupby(['dataset','method','scope']).action_rate.sum(),1.)
    for row in family_summary.groupby(['dataset','method','scope'])[['helpful_rate_all_windows','harmful_rate_all_windows']].sum().reset_index().itertuples():
        expected=REFERENCE['aggregate_summary'][(REFERENCE['aggregate_summary'].dataset==row.dataset)&(REFERENCE['aggregate_summary'].method==row.method)&(REFERENCE['aggregate_summary'].scope==row.scope)].iloc[0]
        assert abs(row.helpful_rate_all_windows-expected.helpful_rate)<1e-12
        assert abs(row.harmful_rate_all_windows-expected.harmful_rate)<1e-12
    for name,frame in [('fault_severity_summary',by_severity),('fault_family_summary',by_family),
                       ('action_family_per_condition',family),('action_family_summary',family_summary)]:
        save_csv(OUTPUT/f'notebook15_{name}.csv',frame)
    worst=harm[(harm.method==PRIMARY)].copy().sort_values(['dataset','scope','maximum_condition_harm_pct'],ascending=[True,True,False])
    save_csv(OUTPUT/'notebook15_primary_failure_audit.csv',worst)
    return by_severity,family_summary


LABELS={'keep':'Keep','residual_gate_mask':'Residual mask','confidence_select':'Confidence selection',
 'harm_aware_policy':'Original harm-aware policy','oof_accuracy_utility':'Pooled accuracy utility','oof_f1_utility':'Pooled F1 utility',
 'window_mean_f1':'Matched window ensemble','window_min_f1':'Matched window minimum','source_min_f1':'Source minimum',
 'gated_fusion_safe':'Gated fusion','residual_stacking_safe':'Residual stacking (guarded)',
 'residual_stacking_accuracy':'Residual stacking (accuracy)','source_mean_f1':'Source ensemble (ours)'}
DS_LABELS={'dsads':'DSADS','har':'HAR'}
SCOPE_LABELS={'clean':'Clean','known_faults':'Known faults','stuck_excluded_from_fitting':'Stuck'}


def latex_escape(value):
    return str(value).replace('\\',r'\textbackslash{}').replace('&',r'\&').replace('%',r'\%').replace('_',r'\_').replace('#',r'\#')


def latex_table(path,headers,rows,caption,label,alignment):
    content=['% Requires \\usepackage{booktabs}',r'\begin{table*}[t]',r'\centering',r'\small',
      r'\caption{'+caption+'}',r'\label{'+label+'}',r'\begin{tabular}{'+alignment+'}',r'\toprule',
      ' & '.join(headers)+r' \\',r'\midrule']
    content += [' & '.join(row)+r' \\' for row in rows]
    content += [r'\bottomrule',r'\end{tabular}',r'\end{table*}','']
    path.write_text('\n'.join(content))


def paper_tables(aggregate,comparison,harm):
    paper=OUTPUT/'paper';paper.mkdir(exist_ok=True)
    columns=[('dsads','known_faults'),('dsads','stuck_excluded_from_fitting'),('har','known_faults'),('har','stuck_excluded_from_fitting')]
    lookup=aggregate.set_index(['dataset','method','scope'])
    full=[]
    for method in METHODS:
        row={'method':method,'diagnostic_oracle':method=='oracle_diagnostic'}
        for ds,scope in columns:
            record=lookup.loc[ds,method,scope];prefix=ds+'_'+scope
            row[prefix+'_mean_pct']=record.macro_f1*100;row[prefix+'_seed_sd_pp']=record.macro_f1_seed_std*100
        full.append(row)
    full=pd.DataFrame(full);save_csv(paper/'all_methods_table.csv',full)
    def write_score_table(methods,name,caption):
        rows=[]
        for method in methods:
            cells=[latex_escape(LABELS.get(method,method))]
            for ds,scope in columns:
                record=lookup.loc[ds,method,scope]
                value=f'{record.macro_f1*100:.2f} $\\pm$ {record.macro_f1_seed_std*100:.2f}'
                cells.append(value)
            rows.append(cells)
        latex_table(paper/(name+'.tex'),['Method','DSADS known','DSADS stuck','HAR known','HAR stuck'],rows,caption,'tab:'+name,'lrrrr')
        save_csv(paper/(name+'.csv'),full[full.method.isin(methods)].set_index('method').loc[methods].reset_index())
    write_score_table(DISPLAY_METHODS,'main_results',
      'Exploratory frozen-test Macro-F1 (percent). Values are the mean and descriptive standard deviation of three seed means. Required folds are averaged equally within seed. Stuck faults were excluded from fitting, but previous test outcomes were inspected.')
    write_score_table(['keep','harm_aware_policy']+NEW_METHODS,'controller_ablation',
      'Controller comparisons with the original aggregation. The matched window ensemble controls head capacity and training-row counts for the source ensemble. Pooled controllers are additional baselines, rather than capacity-matched controls.')
    selected=comparison[comparison.comparator.isin(['keep','oof_accuracy_utility','window_mean_f1','residual_stacking_safe'])&comparison.scope.isin(['known_faults','stuck_excluded_from_fitting'])]
    rows=[]
    for row in selected.itertuples():
        rows.append([DS_LABELS[row.dataset],SCOPE_LABELS[row.scope],latex_escape(LABELS[row.comparator]),
          f'{row.secondary_participant_balanced_delta_pp:+.2f}',f'[{row.secondary_ci_low_pp:+.2f}, {row.secondary_ci_high_pp:+.2f}]'])
    latex_table(paper/'secondary_participant_intervals.tex',['Dataset','Scope','Comparator',r'$\Delta$ F1 (pp)',r'Conditional 95\% interval'],rows,
      'Secondary participant-balanced Macro-F1 differences. Participants are paired and resampled as whole clusters, retaining all seeds and fault views. Intervals are conditional on saved fits and do not apply to primary scores or support significance or population-risk claims.',
      'secondary_participant_intervals','lllrr')
    save_csv(paper/'secondary_participant_intervals.csv',selected)
    rows=[];harm_rows=[]
    for (ds,scope),x in harm[harm.method==PRIMARY].groupby(['dataset','scope'],sort=False):
        avg=float(x.mean_harm_pct.mean());worst=float(x.maximum_condition_harm_pct.max());bad=int((~x.mean_harm_le_1pct).sum())
        rows.append([DS_LABELS[ds],SCOPE_LABELS[scope],f'{avg:.3f}',f'{worst:.3f}',str(bad)+'/'+str(len(x))])
        harm_rows.append({'dataset':ds,'scope':scope,'mean_harm_pct':avg,'worst_condition_harm_pct':worst,'runs_exceeding_1pct_mean_harm':bad,'runs':len(x)})
    latex_table(paper/'primary_harm.tex',['Dataset','Scope',r'Mean harm (\%)',r'Worst condition (\%)',r'Runs above 1\%'],rows,
      'Observed correct-to-incorrect changes as a percentage of all test windows. Validation guards do not guarantee test harm bounds. Worst-condition and run-level failures are retained.',
      'primary_harm','llrrr')
    save_csv(paper/'primary_harm.csv',pd.DataFrame(harm_rows))
    return paper


COLORS={'keep':'#777777','harm_aware_policy':'#009E73','oof_accuracy_utility':'#0072B2',
 'window_mean_f1':'#CC79A7','gated_fusion_safe':'#56B4E9','residual_stacking_safe':'#E69F00',PRIMARY:'#D55E00'}


def save_figure(fig,paper,name):
    fig.savefig(paper/(name+'.pdf'),bbox_inches='tight',metadata={'Creator':'SensorRepair saved-prediction audit'})
    fig.savefig(paper/(name+'.png'),dpi=300,bbox_inches='tight')
    plt.close(fig)


def paper_figures(paper,severity,aggregate,participants):
    plt.rcParams.update({'font.family':'DejaVu Serif','font.size':8,'axes.labelsize':8,'axes.titlesize':8.5,
      'legend.fontsize':7,'xtick.labelsize':7,'ytick.labelsize':7,'axes.spines.top':False,'axes.spines.right':False,
      'pdf.fonttype':42,'ps.fonttype':42,'savefig.facecolor':'white'})
    methods=['keep','oof_accuracy_utility','window_mean_f1','residual_stacking_safe',PRIMARY]
    fig,axes=plt.subplots(2,4,figsize=(7.2,4.0),sharey='row')
    handles=[]
    for i,ds in enumerate(['dsads','har']):
        for j,fault in enumerate(['dropout','bias','gain','stuck']):
            ax=axes[i,j]
            for method in methods:
                x=severity[(severity.dataset==ds)&(severity.fault_family==fault)&(severity.method==method)].sort_values('severity_level')
                line,=ax.plot(x.severity_level,x.macro_f1*100,marker='o',markersize=2.8,lw=1.3,color=COLORS[method],label=LABELS[method])
                if i==0 and j==0:handles.append(line)
            ax.set_xticks([1,2,3]);ax.grid(axis='y',alpha=.18);ax.yaxis.set_major_locator(MaxNLocator(5))
            if i==0:ax.set_title(fault.capitalize())
            if j==0:ax.set_ylabel(DS_LABELS[ds]+' Macro-F1 (%)')
            if i==1:ax.set_xlabel('Severity level')
    fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,1.01),ncol=3,frameon=False)
    fig.subplots_adjust(top=.80,bottom=.13,left=.09,right=.99,hspace=.36,wspace=.16)
    save_figure(fig,paper,'figure01_fault_robustness')
    methods=['keep','harm_aware_policy','oof_accuracy_utility','window_mean_f1','gated_fusion_safe','residual_stacking_safe',PRIMARY]
    fig,axes=plt.subplots(2,2,figsize=(7.2,5.0))
    markers=['o','s','D','v','P','^','*']
    for i,ds in enumerate(['dsads','har']):
        for j,scope in enumerate(['known_faults','stuck_excluded_from_fitting']):
            ax=axes[i,j]
            for method,marker in zip(methods,markers):
                row=aggregate[(aggregate.dataset==ds)&(aggregate.method==method)&(aggregate.scope==scope)].iloc[0]
                ax.scatter(row.harmful_rate*100,row.macro_f1*100,s=80 if method==PRIMARY else 28,
                    color=COLORS[method],marker=marker,edgecolors='white',linewidths=.35,zorder=4 if method==PRIMARY else 3)
            ax.grid(alpha=.18);ax.set_title(DS_LABELS[ds]+': '+SCOPE_LABELS[scope]);ax.set_ylabel('Macro-F1 (%)')
            ax.set_xlabel('Correct-to-incorrect changes (% of windows)')
            ax.xaxis.set_major_locator(MaxNLocator(5));ax.yaxis.set_major_locator(MaxNLocator(5))
            lo,hi=ax.get_xlim();ax.set_xlim(min(-.025,lo),hi)
    handles=[Line2D([],[],marker=m,color=COLORS[k],linestyle='None',markersize=7 if k==PRIMARY else 4.5,label=LABELS[k]) for k,m in zip(methods,markers)]
    fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,1.01),ncol=3,frameon=False)
    fig.subplots_adjust(top=.80,bottom=.12,left=.09,right=.99,hspace=.80,wspace=.25)
    save_figure(fig,paper,'figure02_f1_harm_tradeoff')
    fig,axes=plt.subplots(2,2,figsize=(7.2,4.6))
    for i,ds in enumerate(['dsads','har']):
        for j,scope in enumerate(['known_faults','stuck_excluded_from_fitting']):
            ax=axes[i,j]
            x=participants[(participants.dataset==ds)&(participants.scope==scope)&participants.method.isin([PRIMARY,'window_mean_f1'])]
            p=x.pivot(index=['subject','seed'],columns='method',values='macro_f1_present_classes')
            delta=(p[PRIMARY]-p['window_mean_f1'])*100
            mean=delta.groupby('subject').mean();low=delta.groupby('subject').min();high=delta.groupby('subject').max()
            positions=np.arange(len(mean))
            ax.errorbar(positions,mean.to_numpy(),yerr=np.maximum(np.vstack([mean-low,high-mean]),0.),fmt='o',color=COLORS[PRIMARY],
                        ecolor='#999999',elinewidth=1.,capsize=2,markersize=3.5)
            ax.axhline(0,color='#555555',lw=.9,ls='--');ax.set_xticks(positions,mean.index.astype(str))
            ax.set_title(DS_LABELS[ds]+': '+SCOPE_LABELS[scope]);ax.set_xlabel('Test participant')
            ax.set_ylabel('Participant F1 difference (pp)');ax.grid(axis='y',alpha=.18)
    fig.suptitle('Source ensemble versus matched window ensemble',y=.995,fontsize=9)
    fig.subplots_adjust(top=.88,bottom=.13,left=.09,right=.99,hspace=.65,wspace=.25)
    save_figure(fig,paper,'figure03_participant_differences')


def write_notes(paper,cov,diagnostics):
    text='''# SensorRepair: methods and interpretation notes

Primary method: `source_mean_f1`. Every method and configuration is retained. The study is exploratory after previous test inspection. Stuck faults were excluded from fitting, but earlier stuck outcomes were inspected. The method, controller capacity, validation grid and empirical guards were fixed for the repeated-seed study. Each configuration selected settings on its own validation participants. The source jackknife excludes controller outcome rows. It does not independently refit the entire auxiliary pipeline inside each jackknife block.

## Aggregation and uncertainty

Primary: average condition Macro-F1 within scope, average required participant folds within seed, then average the three seed scores. Seed standard deviations describe variation over the three saved runs. They are not confidence intervals.

Secondary: average each participant's present-class Macro-F1 over conditions and seeds, then average participants equally. The bootstrap resamples whole participants, paired across methods, while retaining all seed/fault views within each person. Its percentile intervals are conditional on saved fits and the fixed evaluation design. They do not quantify the primary-score uncertainty, the refitting/selection process, or population risk. Shared training data cause dependence that this fixed-fit bootstrap does not resolve. No significance or multiplicity-adjusted inference is claimed.

Macro-F1 is not decomposed additively by action. Action diagnostics report correct-to-incorrect and incorrect-to-correct transitions, each divided by all windows, together with conditional rates among selected actions. Sensor interventions can preserve the original predicted class. All candidate predictions were previously computed, so these action rates do not establish inference-cost savings.

## Tables and figures

`main_results.tex`: mean and descriptive seed standard deviation, using the primary aggregation. `controller_ablation.tex`: original, pooled, source and matched controllers. `secondary_participant_intervals.tex`: the separate secondary metric. `primary_harm.tex`: average and worst-condition harm. The full CSV includes all 28 methods, with the label-based oracle marked as diagnostic.

Figure 1 averages required folds and seeds at each fault/severity. Severity levels refer to the original protocol below. Figure 2 shows primary aggregate Macro-F1 against mean correct-to-incorrect changes as a percentage of all windows. It is a descriptive tradeoff plot, not a certified frontier. Figure 3 shows participant-balanced differences versus the matched ensemble; bars are the min/max range across three seeds, not confidence intervals.

## Required limitations

- The same eight DSADS and nine HAR test participants are reused across seeds. DSADS training sets overlap across outer folds. The 15 configurations are not 15 independent participant samples.
- The test sets were inspected during development. Repeated seeds cannot restore untouched confirmation.
- Validation harm guards do not imply test guarantees. All per-condition, participant and run failures are retained. Inspect `notebook15_primary_failure_audit.csv` and the harmful-case table.
- HAR improvements may be small, uneven across seeds, and lower than stronger fusion/stacking baselines. Report those methods. A tiny numerical lead over a strong controller is not evidence of significant superiority.
- Matched ensemble results support an empirical comparison under this pipeline. They do not establish universal domain generalization or novelty. Position the precise contribution against published related work before making originality claims.

Use \\usepackage{booktabs} for the LaTeX tables. The tables are components for the manuscript, not a complete submission.
'''
    text+='\n## Verified coverage\n\n'+cov.to_string(index=False)+'\n'
    text+='\n## Observed primary harm diagnostics\n\n'+diagnostics.round(4).to_string(index=False)+'\n'
    text+='\n## Original fault severities\n\n```json\n'+json.dumps(protocol['fault_severities'],indent=2)+'\n```\n'
    (paper/'methods_and_limitations.md').write_text(text)
    save_json(paper/'analysis_plan.json',{'analysis_signature':ANALYSIS_SIGNATURE,'plan':plan})


# Resolve and freeze provenance before opening prediction arrays.
PREPARED=PROJECT/'processed/v1'
ROOTS={i:PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [9,12,13,14]}
SOURCE14=ROOTS[14]
STAGES={f'nb{i:02d}':PROJECT/'outputs'/f'notebook{i:02d}_v1' for i in [2,4,5]}
reports={};manifests={};report_hashes={};manifest_hashes={}
for number in [9,12,13,14]:
    report,rhash=stage_json(number,f'notebook{number:02d}_run_report.json')
    manifest,mhash=stage_json(number,f'notebook{number:02d}_study_manifest.json')
    assert report['study_complete'],f'Finish Notebook {number:02d} first.'
    assert report['study_signature']==manifest['signature']==json_signature(manifest['plan'])
    reports[number]=report;manifests[number]=manifest;report_hashes[f'nb{number:02d}']=rhash;manifest_hashes[f'nb{number:02d}']=mhash
report14=reports[14];manifest14=manifests[14]
assert report14['primary_method']==PRIMARY and report14['methods_reported']==len(METHODS)
assert report14['total_test_configurations_complete']==report14['total_configurations_required']==len(JOBS)
assert manifest14['plan']['methods']==METHODS and manifest14['plan']['seeds']==SEEDS
for stage in [9,12,13]:
    assert manifest14['plan']['source_report_hashes'][f'nb{stage:02d}']==report_hashes[f'nb{stage:02d}']
    assert manifest14['plan']['source_manifest_hashes'][f'nb{stage:02d}']==manifest_hashes[f'nb{stage:02d}']
prepared_inventory_path=PREPARED/'artifact_inventory.json'
assert file_sha256(prepared_inventory_path)==manifest14['plan']['prepared_inventory_sha256']
prepared_inventory={r['filename']:r for r in json.loads(prepared_inventory_path.read_text())}
protocol_path=verified_copy(PREPARED/'protocol.json','prepared/protocol.json',prepared_inventory['protocol.json']['sha256'])
protocol=json.loads(protocol_path.read_text())
assert protocol['model_seeds']==SEEDS and protocol['development_fault_types']==['dropout','bias','gain']
records14={identity(r):r for r in report14['test_results']+report14['imported_results']}
assert set(records14)=={identity(j) for j in JOBS}
new14={identity(r):r for r in report14['test_results']}
new13={identity(r):r for r in reports[13]['test_results']}
old12={identity(r):r for r in reports[12]['test_results']}
INPUT_RUNS={}
for job in JOBS:
    key=identity(job);stage=14 if key in new14 else (13 if key in new13 else 12)
    record=({14:new14,13:new13,12:old12}[stage])[key]
    assert record==records14[key] and record['study_signature']==reports[stage]['study_signature']
    assert any(a['filename']=='test_predictions.npz' for a in record['artifacts'])
    INPUT_RUNS[key]={'stage':stage,'record':record}
CACHE_INPUTS={identity(j):manifests[9]['plan']['cache_inputs'][identity(j)] for j in JOBS}
reference_names=['aggregate_summary','per_seed_summary','test_summary','per_condition','per_participant','comparisons',
 'per_seed_comparisons','per_fold_comparisons','harm_audit','coverage','dataset_coverage','primary_diagnostics']
inventory14={r['filename']:r for r in json.loads((SOURCE14/'notebook14_artifact_inventory.json').read_text())}
REFERENCE_HASHES={name:inventory14['notebook14_'+name+'.csv']['sha256'] for name in reference_names}
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'jobs':JOBS,'methods':METHODS,'primary_method':PRIMARY,
 'source_report_hashes':report_hashes,'source_manifest_hashes':manifest_hashes,'source14_csv_hashes':REFERENCE_HASHES,
 'prediction_inputs':INPUT_RUNS,'candidate_cache_inputs':CACHE_INPUTS,
 'prepared_inventory_sha256':file_sha256(prepared_inventory_path),'scopes':SCOPES,
 'primary_aggregation':report14['aggregation'],'comparators':COMPARATORS,'display_methods':DISPLAY_METHODS,
 'bootstrap':{'resamples':BOOTSTRAP_RESAMPLES,'seed':BOOTSTRAP_SEED,'confidence_level':CONFIDENCE_LEVEL,
   'unit':'whole participant, paired across methods; retain all seeds and fault views',
   'secondary_metric':'mean participant present-class Macro-F1, averaged over conditions and seeds first',
   'conditional_on_saved_fits':True,'applies_to_primary_score':False,'significance_claim':False},
 'candidate_action_mapping_verified':True,'new_training':0,'new_model_inference':0,'new_test_decisions':0,'threshold_changes':0,
 'test_status':report14['test_status'],'jackknife_limitation':report14['jackknife_limitation'],
 'software':{'numpy':np.__version__,'pandas':pd.__version__,'matplotlib':matplotlib.__version__}}
ANALYSIS_SIGNATURE=json_signature(plan)
path=OUTPUT/'notebook15_analysis_manifest.json'
if path.exists():assert json.loads(path.read_text())['signature']==ANALYSIS_SIGNATURE,'Analysis code, inputs or settings changed; keep the original settings.'
else:save_json(path,{'signature':ANALYSIS_SIGNATURE,'plan':plan})
print('Analysis plan recorded:',ANALYSIS_SIGNATURE[:16],flush=True)
REFERENCE={name:table14(name) for name in reference_names}
ANCHORS={};results=[];paused=None
try:
    for job in JOBS:results.append(audit_one(job))
    check_budget()
except PauseAudit as e:
    paused=str(e)

if paused is not None:
    report={'code_version':CODE_VERSION,'analysis_signature':ANALYSIS_SIGNATURE,'study_complete':False,
      'configurations_complete':len(results),'configurations_required':len(JOBS),'pause_reason':paused,
      'new_training':0,'new_model_inference':0,'new_test_decisions':0,'threshold_changes':0,'gpu_required':False}
    save_json(OUTPUT/'notebook15_run_report.json',report)
    print('\nNOTEBOOK 15 PAUSED',flush=True);print(paused,flush=True)
    print(f'Completed audits: {len(results)}/{len(JOBS)} | Outputs: {OUTPUT}',flush=True)
else:
    frames,aggregate,per_seed,comparisons,harm,cov,diagnostics=reproduce_aggregates(results)
    primary=comparisons[comparisons.method==PRIMARY].copy()
    effects,balanced=participant_bootstrap(frames['per_participant'],primary)
    severity,action_family=diagnostic_summaries(frames,harm)
    paper=paper_tables(aggregate,effects,harm)
    paper_figures(paper,severity,aggregate,frames['per_participant'])
    write_notes(paper,cov,diagnostics)
    save_csv(paper/'primary_comparison_summary.csv',effects)
    report={'code_version':CODE_VERSION,'analysis_signature':ANALYSIS_SIGNATURE,'study_complete':True,
      'configurations_complete':len(results),'configurations_required':len(JOBS),'methods_reproduced':len(METHODS),
      'notebook14_summary_tables_reproduced':reference_names,'all_prediction_metrics_reproduced':True,
      'all_candidate_action_mappings_verified':True,'completed_results':results,'dataset_coverage':cov.to_dict('records'),
      'primary_diagnostics':diagnostics.to_dict('records'),'bootstrap':plan['bootstrap'],
      'secondary_intervals_do_not_apply_to_primary':True,'participant_counts_are_not_multiplied_by_seeds':True,
      'new_training':0,'new_model_inference':0,'new_test_decisions':0,'threshold_changes':0,'gpu_required':False,
      'test_status':plan['test_status'],'jackknife_limitation':plan['jackknife_limitation'],
      'original_outputs_modified':False,'significance_or_risk_guarantee':False,'novelty_guarantee':False,
      'paper_bundle':'notebook15_paper_bundle.zip','output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook15_run_report.json',report)
    with zipfile.ZipFile(OUTPUT/'notebook15_paper_bundle.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(paper.iterdir()):
            if p.is_file():archive.write(p,arcname='paper/'+p.name)
        archive.write(OUTPUT/'notebook15_run_report.json',arcname='notebook15_run_report.json')
    print('\nNOTEBOOK 15 COMPLETE',flush=True)
    print('All 28 methods, 15 configurations, Notebook 14 metrics, comparisons and harm audits reproduced.',flush=True)
    print('Saved sensor-action mappings match the original candidate caches.',flush=True)
    view=effects[effects.comparator.isin(DISPLAY_COMPARATORS)&effects.scope.isin(['known_faults','stuck_excluded_from_fitting'])]
    print('\nPRIMARY EFFECTS | original aggregation | percentage points',flush=True)
    print(view[['dataset','scope','comparator','delta_macro_f1_pp','positive_runs','negative_runs','tied_runs']].round(3).to_string(index=False),flush=True)
    view=effects[effects.comparator.isin(['keep','oof_accuracy_utility','window_mean_f1','residual_stacking_safe'])&effects.scope.isin(['known_faults','stuck_excluded_from_fitting'])]
    print('\nSECONDARY PARTICIPANT-BALANCED EFFECTS | separate metric | conditional 95% participant-bootstrap intervals',flush=True)
    print(view[['dataset','scope','comparator','secondary_participants','secondary_participant_balanced_delta_pp','secondary_ci_low_pp','secondary_ci_high_pp']].round(3).to_string(index=False),flush=True)
    print('Secondary intervals do not apply to primary scores and are not significance tests or population guarantees.',flush=True)
    print('\nOBSERVED PRIMARY HARM | all test failures retained',flush=True)
    print(diagnostics.round(3).to_string(index=False),flush=True)
    view=action_family[(action_family.method==PRIMARY)&action_family.scope.isin(['known_faults','stuck_excluded_from_fitting'])].copy()
    for c in ['action_rate','helpful_rate_all_windows','harmful_rate_all_windows']:view[c]*=100
    print('\nPRIMARY ACTION FAMILIES | percentages of all windows | interventions can preserve the predicted class',flush=True)
    print(view[['dataset','scope','action_family','action_rate','helpful_rate_all_windows','harmful_rate_all_windows']].round(3).to_string(index=False),flush=True)
    print('New training/inference/test decisions/threshold changes: 0 | GPU required: No',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('Next: freeze experiments, compare the precise method with related work, and write the manuscript with stronger baselines and limitations.',flush=True)
    print('Share final output, notebook15_comparison_summary.csv, notebook15_harm_audit.csv, notebook15_action_family_summary.csv, notebook15_run_report.json, notebook15_paper_bundle.zip.',flush=True)
inventory=[{'filename':p.name,'sha256':file_sha256(p),'bytes':p.stat().st_size} for p in sorted(OUTPUT.iterdir())
           if p.is_file() and p.name!='notebook15_artifact_inventory.json']
save_json(OUTPUT/'notebook15_artifact_inventory.json',inventory)


Mounted at /content/drive
NOTEBOOK 15 | final saved-prediction audit and paper outputs | CPU | one executable cell
Both datasets, all 15 configurations, all 28 saved methods. No training, model inference or threshold changes.
Primary aggregation unchanged. Participant-bootstrap intervals are secondary and conditional on saved fits.
Analysis plan recorded: 1adda1976159b892
  Reproduced dsads_fold_0_seed_2026: 2280 windows; 2 participants; saved actions verified.
  Reproduced dsads_fold_1_seed_2026: 2280 windows; 2 participants; saved actions verified.
  Reproduced dsads_fold_2_seed_2026: 2280 windows; 2 participants; saved actions verified.
  Reproduced dsads_fold_3_seed_2026: 2280 windows; 2 participants; saved actions verified.
  Reproduced har_fold_0_seed_2026: 2947 windows; 9 participants; saved actions verified.
  Reproduced dsads_fold_0_seed_2027: 2280 windows; 2 participants; saved actions verified.
  Reproduced dsads_fold_1_seed_2027: 2280 windows; 2 participants; saved actions 